### Bronze Layer

The Bronze layer represents the first entry point of data into Delta Lake. Landing Parquet
files are read and written into Delta tables, introducing ACID transaction guarantees, schema
enforcement, and time-travel capability that plain file formats do not provide. An `ingested_at`
timestamp is added to every table to support audit and lineage tracking across the pipeline.

Historical and incremental data are maintained as separate Delta tables per entity, preserving
the distinction between the initial full load and newly arriving data. This structure enables
the Silver layer to apply Delta MERGE semantics — treating historical data as the merge target
and incremental data as the merge source — for update and insert resolution.

Bronze intentionally applies no business logic, deduplication, or type transformation; all
source columns are retained as strings to preserve raw fidelity. Data quality enforcement and
type standardization are addressed in the Silver layer.

In [0]:
from pyspark.sql.functions import current_timestamp, col

spark.sql("CREATE SCHEMA IF NOT EXISTS apex_retail.bronze")

landing_base = "/Volumes/apex_retail/retail/source_files/landing"
entities = ["customer", "product", "sales"]

def to_string_cols(df):
    # keeping everything as string in bronze, casting happens in silver
    return df.select([col(c).cast("string").alias(c) for c in df.columns])

In [0]:
display(spark.sql("SHOW SCHEMAS IN apex_retail"))

databaseName
bronze
default
gold
gold_tables
information_schema
retail
silver


In [0]:
# Historical Load (initial write)
# Overwrite is used here since this is a one-time full load of historical data,
# establishing the baseline Bronze table for each entity.

for entity in entities:
    df = spark.read.parquet(f"{landing_base}/historical/{entity}")

    # ingested_at gives a per-batch audit timestamp, so any record's entry
    # point into the pipeline can be traced later if needed
    df = to_string_cols(df).withColumn("ingested_at", current_timestamp())

    df.write.format("delta").mode("overwrite").saveAsTable(f"apex_retail.bronze.{entity}_historical")
    print(f"{entity} (historical) -> bronze.{entity}_historical, {df.count()} rows")

customer (historical) -> bronze.customer_historical, 1052 rows
product (historical) -> bronze.product_historical, 1043 rows
sales (historical) -> bronze.sales_historical, 1002 rows


In [0]:
for entity in entities:
    hist_cols = set(spark.table(f"apex_retail.bronze.{entity}_historical").columns)
    inc_cols = set(spark.table(f"apex_retail.bronze.{entity}_incremental").columns)

    only_in_hist = hist_cols - inc_cols
    only_in_inc = inc_cols - hist_cols

    print(f"--- {entity} ---")
    print("Columns only in historical:", only_in_hist)
    print("Columns only in incremental:", only_in_inc)

--- customer ---
Columns only in historical: set()
Columns only in incremental: {'is_current', 'effective_start_date', 'version', 'surrogate_key', 'effective_end_date'}
--- product ---
Columns only in historical: set()
Columns only in incremental: {'last_updated'}
--- sales ---
Columns only in historical: set()
Columns only in incremental: set()


In [0]:
# Incremental Load (daily append)
# Append mode simulates how new data would arrive in a real pipeline - as daily
# batches layered on top of history, rather than replacing it.
# Bronze is intentionally append-only with no deduplication or merge logic;
# raw data is kept exactly as received. Deduplication and conflict resolution
# happen in Silver via Delta MERGE, keeping Bronze a clean, unaltered record
# of everything that was ingested.

for entity in entities:
    df = spark.read.parquet(f"{landing_base}/incremental/{entity}")
    df = to_string_cols(df).withColumn("ingested_at", current_timestamp())

    df.write.format("delta").mode("overwrite").saveAsTable(f"apex_retail.bronze.{entity}_incremental")
    print(f"{entity} (incremental) -> bronze.{entity}_incremental, {df.count()} rows")

customer (incremental) -> bronze.customer_incremental, 1053 rows
product (incremental) -> bronze.product_incremental, 1041 rows
sales (incremental) -> bronze.sales_incremental, 1000 rows


In [0]:
for entity in entities:
    spark.sql(f"DROP TABLE IF EXISTS apex_retail.bronze.{entity}")

In [0]:
display(spark.sql("SHOW TABLES IN apex_retail.bronze"))

database,tableName,isTemporary
bronze,customer_historical,false
bronze,customer_incremental,false
bronze,product_historical,false
bronze,product_incremental,false
bronze,sales_historical,false
bronze,sales_incremental,false


In [0]:
for entity in entities:
    h_count = spark.table(f"apex_retail.bronze.{entity}_historical").count()
    i_count = spark.table(f"apex_retail.bronze.{entity}_incremental").count()
    print(f"{entity}: historical={h_count}, incremental={i_count}, total={h_count + i_count}")

customer: historical=1052, incremental=1053, total=2105
product: historical=1043, incremental=1041, total=2084
sales: historical=1002, incremental=1000, total=2002


In [0]:
display(spark.table("apex_retail.bronze.sales_historical").limit(10))
spark.table("apex_retail.bronze.sales_historical").printSchema()

transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend,ingested_at
503290,2020-10-11 10:08:52,1,1480,8,49.72,0.5,Credit Card,Location A,18,Wednesday,27,7,563.16,271,20% Off,No,Spring,Yes,2026-07-11T19:53:16.561Z
347796,2021-12-08 01:07:40,2,1597,7,817.76,0.32,Credit Card,Location C,15,Friday,20,2,7554.57,631,Flash Sale,No,Summer,Yes,2026-07-11T19:53:16.561Z
493688,2020-02-17 09:40:48,3,5142,8,270.3,0.35,Debit Card,Location A,9,Saturday,35,6,7564.14,879,Flash Sale,Yes,Winter,Yes,2026-07-11T19:53:16.561Z
861348,2020-08-13 00:43:14,4,8447,2,547.84,0.1,null,Location A,13,Friday,42,8,8125.92,211,Buy One Get One Free,Yes,Winter,No,2026-07-11T19:53:16.561Z
535835,2021-07-02 11:59:03,5,6025,4,785.29,0.17,Mobile Payment,Location C,17,Monday,37,3,114.32,862,Flash Sale,Yes,Summer,Yes,2026-07-11T19:53:16.561Z
978720,2020-12-01 16:31:54,6,1883,2,751.17,0.32,Credit Card,Location B,19,Saturday,50,2,3372.17,171,Buy One Get One Free,Yes,Summer,No,2026-07-11T19:53:16.561Z
24070,2020-03-14 22:58:48,7,7781,5,340.07,0.41,Debit Card,Location A,15,Thursday,12,10,1322.64,664,Buy One Get One Free,No,Spring,No,2026-07-11T19:53:16.561Z
752282,2021-11-17 15:56:04,8,8642,7,17.7,0.42,Debit Card,Location D,22,Saturday,3,9,1716.65,733,Buy One Get One Free,Yes,Summer,Yes,2026-07-11T19:53:16.561Z
11898,2020-05-18 08:48:18,9,7193,7,458.85,0.2,Mobile Payment,Location B,19,Sunday,6,10,1358.62,429,Flash Sale,Yes,Winter,No,2026-07-11T19:53:16.561Z
321956,2020-03-10 18:03:08,10,7739,4,612.28,0.1,Credit Card,Location C,19,Saturday,33,8,6757.7,516,20% Off,Yes,Spring,Yes,2026-07-11T19:53:16.561Z


root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_applied: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- transaction_hour: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- week_of_year: string (nullable = true)
 |-- month_of_year: string (nullable = true)
 |-- total_sales: string (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- promotion_type: string (nullable = true)
 |-- holiday_season: string (nullable = true)
 |-- season: string (nullable = true)
 |-- weekend: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
print("Row count:", spark.table("apex_retail.bronze.sales_incremental").count())
display(spark.table("apex_retail.bronze.sales_incremental").limit(10))

Row count: 1000


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend,ingested_at
5002714,2023-02-04 19:23:44,334,258,8,987.33,0.35,Credit Card,null,19,Monday,5,2,null,100,20% Off,No,Fall,No,2026-07-11T19:53:28.631Z
5256471,2023-02-15 02:02:05,73,5708,2,519.75,0.41,Cash,Location D,2,Monday,7,2,613.31,358,null,null,Summer,null,2026-07-11T19:53:28.631Z
6989689,2023-09-05 07:34:41,195,8719,2,806.52,0.47,Mobile Payment,Location A,7,null,36,9,854.91,475,20% Off,null,null,Yes,2026-07-11T19:53:28.631Z
3145707,2023-11-20 04:12:02,831,5830,7,193.05,0.35,Credit Card,null,4,Thursday,47,11,878.38,967,Flash Sale,Yes,null,null,2026-07-11T19:53:28.631Z
7253487,2023-01-26 14:42:18,379,5489,8,null,0.38,Mobile Payment,Location D,14,null,4,1,2821.02,476,Buy One Get One Free,No,Spring,null,2026-07-11T19:53:28.631Z
7583179,null,601,5080,3,239.94,0.48,null,Location C,4,Wednesday,17,8,374.31,160,null,Yes,null,null,2026-07-11T19:53:28.631Z
9670485,2023-05-14 17:49:31,564,650,5,675.61,0.41,Debit Card,Location C,17,Saturday,null,5,1993.05,915,Buy One Get One Free,Yes,Winter,null,2026-07-11T19:53:28.631Z
4558281,2023-01-09 00:58:09,440,1197,1,769.85,0.0,null,Location A,0,Wednesday,2,1,769.85,219,20% Off,Yes,Fall,null,2026-07-11T19:53:28.631Z
9479252,2023-04-29 21:50:19,41,4911,8,113.05,0.4,null,null,21,Tuesday,17,4,542.64,980,null,Yes,Summer,No,2026-07-11T19:53:28.631Z
9973800,2022-03-26 08:38:14,936,6833,2,434.36,0.27,Credit Card,Location A,8,Sunday,12,3,634.17,884,Flash Sale,No,Winter,Yes,2026-07-11T19:53:28.631Z


In [0]:
hist_schema = spark.table("apex_retail.bronze.sales_historical").schema
inc_schema = spark.table("apex_retail.bronze.sales_incremental").schema

print("Schemas match:", hist_schema == inc_schema)

Schemas match: True


In [0]:
# Confirm actual column names before building DQ rules and MERGE logic against them
spark.table("apex_retail.bronze.customer_historical").printSchema()
spark.table("apex_retail.bronze.product_historical").printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- product_review_count: string (nullable = true)
 |-- product_stock: string (nullable = true)


In [0]:
df = spark.read.parquet(f"{landing_base}/incremental/customer")
df = to_string_cols(df).withColumn("ingested_at", current_timestamp())
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("apex_retail.bronze.customer_incremental")
print("rebuilt, rows:", df.count())
spark.table("apex_retail.bronze.customer_incremental").printSchema()

rebuilt, rows: 1053
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp, col

landing_base = "/Volumes/apex_retail/retail/source_files/landing"

def to_string_cols(df):
    return df.select([col(c).cast("string").alias(c) for c in df.columns])

df = spark.read.parquet(f"{landing_base}/incremental/customer")
df = to_string_cols(df).withColumn("ingested_at", current_timestamp())
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("apex_retail.bronze.customer_incremental")
print("rebuilt, rows:", df.count())
spark.table("apex_retail.bronze.customer_incremental").printSchema()

rebuilt, rows: 1053
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import current_timestamp, col

landing_base = "/Volumes/apex_retail/retail/source_files/landing"

def to_string_cols(df):
    return df.select([col(c).cast("string").alias(c) for c in df.columns])

df = spark.read.parquet(f"{landing_base}/incremental/customer")
print("Source parquet columns:", df.columns)

df = to_string_cols(df).withColumn("ingested_at", current_timestamp())
print("Columns before write:", df.columns)

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("apex_retail.bronze.customer_incremental")
print("Write completed.")

fresh_check = spark.read.table("apex_retail.bronze.customer_incremental")
print("Columns after write (fresh read):", fresh_check.columns)

Source parquet columns: ['customer_id', 'age', 'gender', 'income_bracket', 'loyalty_program', 'membership_years', 'churned', 'marital_status', 'number_of_children', 'education_level', 'occupation', 'customer_zip_code', 'customer_city', 'customer_state']
Columns before write: ['customer_id', 'age', 'gender', 'income_bracket', 'loyalty_program', 'membership_years', 'churned', 'marital_status', 'number_of_children', 'education_level', 'occupation', 'customer_zip_code', 'customer_city', 'customer_state', 'ingested_at']
Write completed.
Columns after write (fresh read): ['customer_id', 'age', 'gender', 'income_bracket', 'loyalty_program', 'membership_years', 'churned', 'marital_status', 'number_of_children', 'education_level', 'occupation', 'customer_zip_code', 'customer_city', 'customer_state', 'ingested_at']


In [0]:
spark.table("apex_retail.bronze.sales_incremental").printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- discount_applied: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- store_location: string (nullable = true)
 |-- transaction_hour: string (nullable = true)
 |-- day_of_week: string (nullable = true)
 |-- week_of_year: string (nullable = true)
 |-- month_of_year: string (nullable = true)
 |-- total_sales: string (nullable = true)
 |-- promotion_id: string (nullable = true)
 |-- promotion_type: string (nullable = true)
 |-- holiday_season: string (nullable = true)
 |-- season: string (nullable = true)
 |-- weekend: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)



### Summary

This notebook writes the Landing Parquet files into Delta Lake tables, the first point in the
pipeline where data gets ACID transaction guarantees and schema enforcement. An `ingested_at`
timestamp was added to every table for audit and lineage tracking.

Historical and incremental data are kept as separate Delta tables per entity (e.g.
customer_historical, customer_incremental) rather than combined - this gives the Silver layer
a clean source and target to run Delta MERGE against. No deduplication or business logic is
applied here; Bronze is intentionally a faithful, minimally-processed copy of Landing data.

**Output:** 6 Delta tables under `apex_retail.bronze` - customer, product, and sales, each
split into historical and incremental, ready for cleansing and merging in the Silver phase.